In [48]:
#Imports
import pandas as pd

from nltk import word_tokenize, sent_tokenize

from gensim.models import Word2Vec
import gensim

import unicodedata as ud
from transformers import pipeline

In [49]:
#Load the model
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"], engine='fastparquet')
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"], engine='fastparquet')
print(f"Train size: {len(df_train)}, Validation size: {len(df_val)}")

Train size: 15343, Validation size: 3011


In [50]:
# Split train and validation into ar, ko and te based on the lang column
df_ar_train = df_train[df_train['lang'] == 'ar']
df_ko_train = df_train[df_train['lang'] == 'ko']
df_te_train = df_train[df_train['lang'] == 'te']

df_ar_val = df_val[df_val['lang'] == 'ar']
df_ko_val = df_val[df_val['lang'] == 'ko']
df_te_val = df_val[df_val['lang'] == 'te']

In [51]:
# --- Setup translation pipeline ---
# Load the Hugging Face NLLB model
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M")

def translate_text(text: str, src_lang: str = "arb_Arab", tgt_lang: str = "eng_Latn") -> str:
    """
    Translate text from source to target language using NLLB.
    Default: Arabic -> English
    """
    if not isinstance(text, str) or text.strip() == "":
        return text
    result = translator(text, src_lang=src_lang, tgt_lang=tgt_lang, max_length=1024)
    return result[0]["translation_text"]
    

Device set to use mps:0


In [52]:
def tokenize_text(input):
    # Remove all punctuation characters, keeping in mind that arabic is written from right to left
    input = ''.join([char for char in input if not ud.category(char).startswith('P')])
    # Tokenize the input into words
    words = word_tokenize(input)
    return words

In [53]:
question = df_ar_train.iloc[0]['question']
trans_question = translate_text(question)
print("Question:\n", question, "\n")
print("Translated question:\n", trans_question, "\n")

tokenised = tokenize_question(translate_text(df_ar_train.iloc[0]['question']))
print("Tokenised output:\n",tokenised, "\n")


Question:
 متى تدخلت روسيا في  الحرب الأهلية السورية؟ 

Translated question:
 When did Russia intervene in the Syrian civil war? 

Tokenised output:
 ['When', 'did', 'Russia', 'intervene', 'in', 'the', 'Syrian', 'civil', 'war'] 



In [72]:
data = []
for i in range(50):
    question = df_ar_train.iloc[i]['question']
    trans_q = translate_text(question)
    tokens = tokenize_text(trans_q)
    data.append(tokens)

In [92]:
data_ar = []
for i in range(50):
    question = df_ar_train.iloc[i]['question']
    tokens = tokenize_text(question)
    data_ar.append(tokens)

In [93]:
len(data)
len(data_ar)

50

In [74]:
w2v = Word2Vec(
    sentences=data,   # your list of lists of words
    vector_size=100,  # dimensionality of word embeddings
    window=5,         # context window size
    min_count=1,      # ignore words that appear less than this
    workers=4,        # number of CPU cores to use
    sg=0              # training algorithm: 0=CBOW, 1=Skip-Gram
)

In [85]:
w2v.wv.most_similar("Israel", topn=5)


[('is', 0.31368178129196167),
 ('control', 0.23489554226398468),
 ('on', 0.22646257281303406),
 ('invasion', 0.21053996682167053),
 ('armies', 0.18817657232284546)]

In [86]:
w2v.wv.most_similar("Russia", topn=5)


[('Forces', 0.2752993106842041),
 ('What', 0.24961404502391815),
 ('death', 0.21611915528774261),
 ('generate', 0.21364718675613403),
 ('free', 0.2130969613790512)]

In [87]:
w2v.wv.most_similar("Jordan", topn=5)


[('Kentucky', 0.28043612837791443),
 ('Forces', 0.2757696807384491),
 ('its', 0.24299408495426178),
 ('reputation', 0.23849640786647797),
 ('peace', 0.2196391075849533)]

In [94]:
w2v_ar = Word2Vec(
    sentences=data,   # your list of lists of words
    vector_size=100,  # dimensionality of word embeddings
    window=5,         # context window size
    min_count=1,      # ignore words that appear less than this
    workers=4,        # number of CPU cores to use
    sg=0              # training algorithm: 0=CBOW, 1=Skip-Gram
)

In [97]:
w2v_ar.wv.most_similar("السورية", topn=5)

[('still', 0.29765066504478455),
 ('العظمى', 0.2655729353427887),
 ('دول', 0.2653824985027313),
 ('كان', 0.25200989842414856),
 ('Saudi', 0.21797475218772888)]